In [1]:
#about torch...
import torch
import torch.nn as nn
import torch
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset

#using numpy
import numpy as np

#for data load or save
import pandas as pd

#visualize some datasets
import matplotlib.pyplot as plt

#check our work directory
import os

#to unzip datasets
import zipfile


In [2]:
os.listdir('../input/dogs-vs-cats-redux-kernels-edition')


['cat.1714.jpg',
 'cat.10025.jpg',
 'dog.3113.jpg',
 'cat.10782.jpg',
 'cat.776.jpg',
 '824.jpg',
 '456.jpg',
 'dog.2753.jpg',
 '1528.jpg',
 'cat.3041.jpg',
 'dog.10802.jpg',
 'dog.10470.jpg',
 'dog.4118.jpg',
 'dog.1646.jpg',
 'cat.2601.jpg',
 'dog.9098.jpg',
 'dog.5758.jpg',
 'dog.11230.jpg',
 'cat.317.jpg',
 'cat.11204.jpg',
 'dog.2495.jpg',
 'dog.789.jpg',
 '790.jpg',
 'dog.2332.jpg',
 '1149.jpg',
 'cat.10836.jpg',
 'cat.10444.jpg',
 'cat.4859.jpg',
 'cat.1375.jpg',
 'dog.3572.jpg',
 'dog.3900.jpg',
 'cat.2260.jpg',
 'dog.5339.jpg',
 'cat.7299.jpg',
 'cat.3852.jpg',
 'cat.3420.jpg',
 'dog.1580.jpg',
 'cat.3387.jpg',
 'dog.10011.jpg',
 'dog.1227.jpg',
 'dog.4579.jpg',
 'dog.1923.jpg',
 'dog.1551.jpg',
 'cat.3883.jpg',
 'dog.8268.jpg',
 'dog.10767.jpg',
 'cat.3356.jpg',
 'cat.6408.jpg',
 'dog.11127.jpg',
 'dog.9428.jpg',
 'cat.7248.jpg',
 'cat.2964.jpg',
 'cat.2516.jpg',
 'cat.10495.jpg',
 'cat.4888.jpg',
 'dog.6928.jpg',
 'dog.3204.jpg',
 'cat.10332.jpg',
 'cat.1403.jpg',
 'cat.1871

In [3]:
os.makedirs('../data', exist_ok=True)


In [4]:
base_dir = '../input/dogs-vs-cats-redux-kernels-edition'
train_dir = '../data/train'
test_dir = '../data/test'


In [5]:
with zipfile.ZipFile(os.path.join(base_dir, 'train.zip')) as train_zip:
    train_zip.extractall('../data')
    
with zipfile.ZipFile(os.path.join(base_dir, 'test.zip')) as test_zip:
    test_zip.extractall('../data')


In [6]:
os.listdir(train_dir)[:5]


['dog', 'train', 'cat']

In [7]:
import glob

train_list = glob.glob(os.path.join(train_dir,'*.jpg'))
test_list = glob.glob(os.path.join(test_dir, '*.jpg'))


In [8]:
len(train_list)


0

In [9]:
train_dog_list = glob.glob(os.path.join(train_dir,'dog*.jpg'))
train_cat_list = glob.glob(os.path.join(train_dir,'cat*.jpg'))


In [10]:
print(len(train_dog_list))
len((train_cat_list))


0


0

In [11]:
train_dog = '../data/train/dog'
train_cat = '../data/train/cat'

os.makedirs(train_dog, exist_ok=True)
os.makedirs(train_cat, exist_ok=True)


In [12]:
import shutil


In [13]:
for file in train_dog_list:
    shutil.move(file,train_dog)


In [14]:
for file in train_cat_list:
    shutil.move(file,train_cat)


In [15]:
test_unknown = '../data/test/unknown'


In [16]:
os.makedirs(test_unknown,exist_ok = True)


In [17]:
for file in test_list:
    shutil.move(file,test_unknown)


In [18]:
print(len(os.listdir(train_dog)))
print(len(os.listdir(train_cat)))
print(len(os.listdir(test_unknown)))


11258
11242
2500


In [19]:
import torch
import torchvision
import torchvision.transforms as transforms
model = torchvision.models.resnet50(pretrained = True)
model.fc = torch.nn.Linear(model.fc.in_features,2)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 108MB/s]


In [20]:
model = model.cuda()


In [21]:
def get_labels(dataset):
    if isinstance(dataset, torch.utils.data.Subset):
        return get_labels(dataset.dataset)[dataset.indices]
    else:
        return np.array([img[1] for img in dataset.imgs])


In [22]:
from sklearn.model_selection import StratifiedShuffleSplit


In [23]:
def setup_train_val_split(labels, dryrun=False, seed=0):
    x = np.arange(len(labels))
    y = np.array(labels)
    splitter = StratifiedShuffleSplit(
        n_splits=1, train_size=0.8, random_state=seed
    )
    train_indices, val_indices = next(splitter.split(x, y))

    if dryrun:
        train_indices = np.random.choice(train_indices, 100, replace=False)
        val_indices = np.random.choice(val_indices, 100, replace=False)

    return train_indices, val_indices


In [24]:
def setup_train_val_datasets(data_dir, dryrun=False):
    dataset = torchvision.datasets.ImageFolder(
        os.path.join(data_dir, "train"),
        transform=setup_center_crop_transform(),
    )
    labels = get_labels(dataset)
    train_indices, val_indices = setup_train_val_split(labels, dryrun)

    train_dataset = torch.utils.data.Subset(dataset, train_indices)
    val_dataset = torch.utils.data.Subset(dataset, val_indices)

    return train_dataset, val_dataset


In [25]:
def setup_train_val_loaders(data_dir, batch_size, dryrun=False):
    train_dataset, val_dataset = setup_train_val_datasets(
        data_dir, dryrun=dryrun
    )
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=2,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, num_workers=2
    )
    return train_loader, val_loader


In [26]:
def setup_center_crop_transform():
    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )


In [27]:
train_loader,val_loader = setup_train_val_loaders('/kaggle/data',batch_size = 50,dryrun= False)


FileNotFoundError: Found no valid file for the classes train. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [28]:
def train_1epoch(model, train_loader, lossfun, optimizer):
    model.train()
    total_loss, total_acc = 0.0, 0.0

    for x, y in tqdm(train_loader):
        x = x.cuda()
        y = y.cuda()

        optimizer.zero_grad()
        out = model(x)
        loss = lossfun(out, y)
        _, pred = torch.max(out.detach(), 1)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_acc += torch.sum(pred == y)

    avg_loss = total_loss / len(train_loader.dataset)
    avg_acc = total_acc / len(train_loader.dataset)
    return avg_acc, avg_loss


In [29]:
def validate_1epoch(model, val_loader, lossfun):
    model.eval()
    total_loss, total_acc = 0.0, 0.0

    with torch.no_grad():
        for x, y in tqdm(val_loader):
            x = x.cuda()
            y = y.cuda()

            out = model(x)
            loss = lossfun(out.detach(), y)
            _, pred = torch.max(out, 1)

            total_loss += loss.item() * x.size(0)
            total_acc += torch.sum(pred == y)

    avg_loss = total_loss / len(val_loader.dataset)
    avg_acc = total_acc / len(val_loader.dataset)
    return avg_acc, avg_loss


In [30]:
from tqdm import tqdm


In [31]:
def train(model, optimizer, train_loader, val_loader, n_epochs):
    lossfun = torch.nn.CrossEntropyLoss()

    for epoch in tqdm(range(n_epochs)):
        train_acc, train_loss = train_1epoch(
            model, train_loader, lossfun, optimizer
        )
        val_acc, val_loss = validate_1epoch(model, val_loader, lossfun)
        print(
            f"epoch={epoch}, train loss={train_loss}, train accuracy={train_acc}, val loss={val_loss}, val accuracy={val_acc}"
        )


In [32]:
train(model,optim.SGD(model.parameters(),lr = 0.01),train_loader,val_loader,n_epochs = 1)


NameError: name 'train_loader' is not defined

In [33]:
def setup_test_loader(data_dir, batch_size, dryrun):
    dataset = torchvision.datasets.ImageFolder(
        os.path.join(data_dir, "test"), transform=setup_center_crop_transform()
    )
    image_ids = [
        os.path.splitext(os.path.basename(path))[0] for path, _ in dataset.imgs
    ]

    if dryrun:
        dataset = torch.utils.data.Subset(dataset, range(0, 100))
        image_ids = image_ids[:100]

    loader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, num_workers=2
    )
    return loader, image_ids


In [34]:
test_loader,image_ids = setup_test_loader('/kaggle/data',batch_size = 50,dryrun = False)


FileNotFoundError: Found no valid file for the classes test. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [35]:
def predict(model, loader):
    pred_fun = nn.Softmax(dim=1)
    preds = []
    for x,_ in tqdm(loader):
        x = x.cuda()
        with torch.no_grad():
            y = pred_fun(model(x))
        y = y.cpu().numpy()
        y = y[:, 1]  # cat:0, dog: 1
        preds.append(y)
    preds = np.concatenate(preds)
    return preds


In [36]:
preds = predict(model,test_loader)


NameError: name 'test_loader' is not defined

In [37]:
def write_prediction(image_ids, prediction, out_path):
    with open(out_path, "w") as f:
        f.write("id,label\n")
        for i, p in zip(image_ids, prediction):
            f.write("{},{}\n".format(i, p))


In [38]:
write_prediction(image_ids,preds,'/kaggle/working/submission.csv')


NameError: name 'image_ids' is not defined